In [3]:
from llm.model import GPTConfig
print(GPTConfig(vocab_size=60))
print(GPTConfig(vocab_size=60, n_layer=2).n_layer)

GPTConfig(vocab_size=60, block_size=64, n_layer=4, n_head=4, n_embd=64, dropout=0.2)
2


In [5]:
import numpy as np
from llm.tokenizer import CharTokenizer

text = open("data/fables.txt", encoding="utf-8").read()
tok = CharTokenizer(text)
data = np.array(tok.encode(text))

x_ids, y_ids = data[0:8], data[1:9]
for t in range(8):
    
    seen = tok.decode(x_ids[:t+1]  )
    target = tok.decode([y_ids[t]])
    print(f"sees {seen!r:12} -> should predict {target!r}")

sees 'T'          -> should predict 'H'
sees 'TH'         -> should predict 'E'
sees 'THE'        -> should predict ' '
sees 'THE '       -> should predict 'F'
sees 'THE F'      -> should predict 'O'
sees 'THE FO'     -> should predict 'X'
sees 'THE FOX'    -> should predict ' '
sees 'THE FOX '   -> should predict 'A'


In [ ]:
from llm.nn import manual_seed

from llm.model import GPT, GPTConfig
manual_seed(0)

model = GPT(GPTConfig(vocab_size=tok.vocab_size))
model.eval()

idx = np.stack([data[0:8], data[100:108]])
x = model.tok_emb(idx) + model.pos_emb(np.arange(8))

print("Embeddings ", x.shape)

for i, block in enumerate(model.blocks):
    x = block(x)
    print(f"Block {i} output shape: {x.shape}")
logits = model.head(model.ln_f(x))
print("Logits ", logits.shape)

Embeddings  (2, 8, 64)
Block 0 output shape: (2, 8, 64)
Block 1 output shape: (2, 8, 64)
Block 2 output shape: (2, 8, 64)
Block 3 output shape: (2, 8, 64)
Logits  (2, 8, 60)


In [7]:
print(f"total: {model.num_params():,}")
C = 64
attn = 4 * ( C* C + C)
mlp =  (C * 4 * C + 4 * C) + (4 * C * C + C)
ln = 2 * (2 * C)

print("One block by formula: ", attn + mlp + ln, " counted: ", model.blocks[0].num_params())


total: 211,840
One block by formula:  49984  counted:  49984


In [8]:
targets = np.stack([data[1:9], data[101:109]])
logits, loss = model(idx, targets)
print(f"starting loss {float(loss.data):.3f}log V = {np.log(60):.3f}")

starting loss 4.094log V = 4.094


In [2]:
import numpy as np
from llm.tensor import numerical_grad
from llm.nn import manual_seed
from llm.model import GPT, GPTConfig

manual_seed(1)
tiny = GPT(GPTConfig(vocab_size=7, block_size=5, n_layer=2,
                     n_head=2, n_embd=8, dropout=0.0))
rng = np.random.default_rng(0)
for p in tiny.parameters():
    p.data = p.data + rng.normal(size=p.shape) * 0.3     # float64
x = rng.integers(0, 7, size=(2, 5))
y = rng.integers(0, 7, size=(2, 5))

tiny(x, y)[1].backward()
loss = lambda: float(tiny(x, y)[1].data)
worst = 0.0
for i, p in enumerate(tiny.parameters()):
    num = numerical_grad(loss, p.data)
    if np.abs(num).max() < 1e-12:        # key biases: gradient is 0
        continue
    err = np.abs(p.grad - num).max() / np.abs(num).max()
    worst = max(worst, err)
    print(i, p.shape, f"{err:.1e}")
print("worst relative error:", worst)

0 (7, 8) 4.0e-09
1 (5, 8) 3.8e-09
2 (8,) 2.6e-09
3 (8,) 5.5e-09
4 (8, 8) 7.8e-09
5 (8,) 1.5e-08
6 (8, 8) 4.8e-09
7 (8,) 1.0e+00
8 (8, 8) 4.9e-09
9 (8,) 8.2e-09
10 (8, 8) 3.2e-09
11 (8,) 5.6e-09
12 (8,) 8.4e-09
13 (8,) 9.2e-09
14 (8, 32) 2.1e-09
15 (32,) 4.8e-09
16 (32, 8) 1.6e-09
17 (8,) 2.7e-09
18 (8,) 2.9e-09
19 (8,) 6.5e-09
20 (8, 8) 6.3e-08
21 (8,) 1.1e-07
22 (8, 8) 1.4e-07
23 (8,) 1.0e+00
24 (8, 8) 6.1e-09
25 (8,) 5.3e-09
26 (8, 8) 4.8e-09
27 (8,) 4.1e-09
28 (8,) 4.2e-09
29 (8,) 7.4e-09
30 (8, 32) 3.1e-09
31 (32,) 4.0e-09
32 (32, 8) 2.2e-09
33 (8,) 1.9e-09
34 (8,) 3.4e-09
35 (8,) 1.2e-09
36 (8, 7) 1.4e-09
worst relative error: 1.0000000048828126


In [12]:
import numpy as np
from llm.tokenizer import CharTokenizer
from llm.nn import manual_seed
from llm.model import GPT, GPTConfig
text = open("data/fables.txt", encoding="utf-8").read()
tok = CharTokenizer(text)
data = np.array(tok.encode(text))
x = np.stack([data[0:32], data[500:532]])
y = np.stack([data[1:33], data[501:533]])
manual_seed(0)
small = GPT(GPTConfig(vocab_size=tok.vocab_size, block_size=32,
n_layer=2, n_head=2, n_embd=32, dropout=0.0))
for step in range(151):
    _, loss = small(x, y)
    small.zero_grad()
    loss.backward()
    for p in small.parameters():
        p.data -= 0.1 * p.grad
    if step % 50 == 0:
        print(f"step {step:3} loss {float(loss.data):.4f}")
    
prediction = small(x)[0].data.argmax(axis=-1)
print(repr(tok.decode(y[0])))
print(repr(tok.decode(prediction[0])))

step   0 loss 4.0943
step  50 loss 4.0883
step 100 loss 4.0356
step 150 loss 3.6658
'HE FOX AND THE WELL\n\nA fox was w'
'                                '


In [1]:
import numpy as np
from llm.nn import manual_seed
from llm.model import GPT, GPTConfig, save_model, load_model
manual_seed(0)
model = GPT(GPTConfig(vocab_size=5, n_layer=1))
model.eval()
save_model(model, list("abcde"), "test_model.npz")
again, chars = load_model("test_model.npz")
ids = np.array([[0, 1, 2]])
print(chars, np.allclose(model(ids)[0].data, again(ids)[0].data))

abcde True


In [2]:
import numpy as np
from llm.tokenizer import CharTokenizer

text = open("data/fables.txt", encoding="utf-8").read()
tok = CharTokenizer(text)
data = np.array(tok.encode(text))


rng = np.random.default_rng(0)
B, T = 3, 12
starts = rng.integers(0, len(data) - T-1, size=B)
x = np.stack([data[s:s+T] for s in starts])
y = np.stack([data[s+1:s+T+1] for s in starts])
print(starts)
for row in range(B):
    print(repr(tok.decode(x[row])), "->", repr(tok.decode(y[row])))

[22322 16715 13413]
'ery young, w' -> 'ry young, we'
' of half the' -> 'of half the '
' will hear t' -> 'will hear th'


In [3]:
import numpy as np
def grad_f(w):
    return np.array([0.1 * w[0], 20 * w[1]])
def run(update, steps=60):
    w, state, path = np.array([-9.0, 1.5]), {}, []
    for t in range(1, steps + 1):
        w = update(w, grad_f(w), state, t)
        path.append(w)
    return np.array(path)

def sgd(w, g, state, t, lr=0.09):
    return w - lr * g


print(run(sgd)[-1])

[-5.23192824e+00  2.29874331e-06]


In [4]:
def momentum(w, g, state, t, lr=0.04, beta=0.9):
    state["v"] = beta * state.get("v", 0) + g
    return w - lr * state["v"]
print(run(momentum)[-1])

[-0.13920244  0.03659652]


In [5]:
def adam(w, g, state, t, lr=0.5, b1=0.9, b2=0.99, eps=1e-8):
    state["m"] = b1 * state.get("m", 0) + (1 - b1) * g
    state["v"] = b2 * state.get("v", 0) + (1 - b2) * g * g
    m_hat = state["m"] / (1 - b1 ** t)
    v_hat = state["v"] / (1 - b2 ** t)
    return w - lr * m_hat / (np.sqrt(v_hat) + eps)

print(run(adam)[-1])

[-0.37347822 -0.05078325]
